# 02 – Build the Matrix

Collaborative filtering works on a **user-item interaction matrix**: rows are users,
columns are movies, and each cell is a rating (or 0 if unrated).

The core idea: users who have rated similar movies probably have similar taste.
We'll find those users and recommend movies they liked that you haven't seen.

This notebook builds that matrix and shows why it's represented as a *sparse* matrix.

In [ ]:
import sys
sys.path.insert(0, "../src")

from rec_sys.data import MovieLensLoader

loader = MovieLensLoader("../data/ml-1m")
loader.load()

## ID Remapping

MovieLens uses 1-based IDs (user 1, movie 1, ...). ALS needs 0-based dense indices
so they can be used directly as row/column indices in a matrix.

The `MovieLensLoader` handles this transparently:
- `user_id_map[movielens_id]` → remapped 0-based index
- `user_id_reverse[remapped_id]` → original MovieLens ID

This round-trips correctly:

In [ ]:
# Pick an example MovieLens user ID and verify the round-trip.
movielens_uid = 42
remapped = loader.remapped_user_id(movielens_uid)
back = loader.original_user_id(int(remapped))

print(f"MovieLens user {movielens_uid} → remapped index {remapped} → back to {back}")
assert back == movielens_uid, "Round-trip failed"
print("Round-trip OK")

## Constructing the Sparse Matrix

A dense matrix for 6,040 users × 3,706 movies = 22.4M cells.
At 4 bytes per float, that's ~90 MB — and 96% of it would be zeros.

We use `scipy.sparse.csr_matrix` instead: stores only the non-zero entries.
Memory usage drops from 90 MB to ~12 MB for the same data.

This matters a lot when you scale to millions of users.

In [ ]:
from scipy import sparse
import numpy as np

# Build coordinate lists: row = user, col = movie, data = rating.
rows = [int(r.user_id) for r in loader.ratings]
cols = [int(r.movie_id) for r in loader.ratings]
data = [r.rating for r in loader.ratings]

n_users = loader.num_users
n_movies = loader.num_items

# Construct sparse matrix in COO format, then convert to CSR.
# COO is easy to build; CSR is efficient for row-slicing (per-user operations).
rating_matrix = sparse.coo_matrix(
    (data, (rows, cols)),
    shape=(n_users, n_movies),
).tocsr()

print(f"Matrix shape     : {rating_matrix.shape}")
print(f"Non-zero entries : {rating_matrix.nnz:,}")
print(f"Density          : {100 * rating_matrix.nnz / (n_users * n_movies):.2f}%")
print(f"Memory (sparse)  : {rating_matrix.data.nbytes / 1e6:.1f} MB")
print(f"Memory (dense)   : {n_users * n_movies * 4 / 1e6:.0f} MB")

## Visualising a Slice

Looking at the full matrix is too big. Let's look at the first 100 users × 100 movies
to get a feel for the sparsity pattern.

Each dot = a rating. The white space = missing data the model must fill in.

In [ ]:
import matplotlib.pyplot as plt

# Extract the first 100x100 block as a dense array for visualisation.
slice_dense = rating_matrix[:100, :100].toarray()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(slice_dense, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_xlabel("Movie index (0-99)")
ax.set_ylabel("User index (0-99)")
ax.set_title("User-Movie Rating Matrix (first 100×100) — white = unrated")
plt.colorbar(im, ax=ax, label="Rating")
plt.tight_layout()
plt.show()

## The Cold-Start Problem Visualised

Some users have many ratings (dense rows); some have very few (sparse rows).
Users with very few ratings are the cold-start problem — not enough signal
for the model to learn reliable preferences.

The popularity fallback from `cold_start.py` handles these users.

In [ ]:
from collections import Counter
from rec_sys.model.cold_start import PopularityFallback
from rec_sys.data.schema import MovieId

ratings_per_user = Counter(int(r.user_id) for r in loader.ratings)

# Cold-start threshold: users with ≤5 ratings.
cold_start_users = [uid for uid, cnt in ratings_per_user.items() if cnt <= 5]
warm_users = [uid for uid, cnt in ratings_per_user.items() if cnt > 5]

print(f"Cold-start users (≤5 ratings): {len(cold_start_users):,} ({100*len(cold_start_users)/len(ratings_per_user):.1f}%)")
print(f"Warm users (>5 ratings)       : {len(warm_users):,} ({100*len(warm_users)/len(ratings_per_user):.1f}%)")

# What would cold-start users get?
popularity = PopularityFallback.compute_popularity(loader.ratings, weighting="hybrid")
top5 = PopularityFallback.top_k_popular(popularity, k=5)
print("\nCold-start fallback (top-5 popular):")
for rank, (mid, score) in enumerate(top5, 1):
    movie = loader.get_movie(MovieId(int(mid)))
    print(f"  {rank}. {movie.title}")

## Key Takeaways

1. **Sparse representation is essential** — storing the full dense matrix would waste 96% of the space.

2. **ID remapping is required** — ALS uses matrix indices directly, so we need contiguous 0-based IDs.

3. **Cold-start is real** — users with few ratings will get poor collaborative recommendations.
   Popularity fallback is a simple, effective baseline.

Next: `03_train_the_model.ipynb` — factorising this matrix with ALS to learn embeddings.